# Alignment pipeline — Qwen3-4B-Instruct-2507

In [ ]:
!pip install -q -U transformers==4.57.1 peft==0.15.2 trl==0.19.0 accelerate==1.6.0 bitsandbytes==0.45.5 datasets==3.2.0 scikit-learn==1.7.2

In [ ]:
import os, json, pickle
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from scipy.sparse import hstack

SEED = 42
set_seed(SEED)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

In [ ]:
REPO_URL = "https://github.com/Vubni/test.git"
if not os.path.exists("data/kid_adult.jsonl"):
    !git clone $REPO_URL repo
    os.chdir("repo")

In [ ]:
def read_jsonl(path):    with open(path) as f:        return [json.loads(l) for l in f]kid_adult = read_jsonl("data/kid_adult.jsonl")public_test_style = read_jsonl("data/public_test_style.jsonl")good_bad = read_jsonl("data/good_bad.jsonl")public_test_quality = read_jsonl("data/public_test_quality.jsonl")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
base_model = prepare_model_for_kbit_training(base_model)

In [ ]:
with open("metrics/style_clf.pkl", "rb") as f:
    style_clf = pickle.load(f)
word_vec, char_vec = style_clf["vecs"]
style_lr = style_clf["clf"]

def p_simple(texts):
    X = hstack([word_vec.transform(texts), char_vec.transform(texts)])
    return style_lr.predict_proba(X)[:, 1]

@torch.no_grad()
def generate(model, prompts, max_new_tokens=256):
    outs = []
    for p in prompts:
        messages = [{"role": "user", "content": p}]
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
        outs.append(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))
    return outs

## Задача 1 — SFT

In [ ]:
def to_text(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["kid"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

sft_dataset = Dataset.from_list(kid_adult).map(to_text, remove_columns=["prompt", "kid", "adult"])

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="adapters/sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    seed=SEED,
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=base_model,
    args=sft_config,
    train_dataset=sft_dataset,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("adapters/sft")

In [ ]:
sft_model = trainer.model
sft_model.eval()

prompts = [d["prompt"] for d in public_test_style]
generations = generate(sft_model, prompts)
scores = p_simple(generations)
print("P_simple (SFT):", scores.mean())

## Задача 2 — DPO по стилю

In [ ]:
import gcdel trainer, sft_modelgc.collect()torch.cuda.empty_cache()

In [ ]:
bf16_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")sft_merged = PeftModel.from_pretrained(bf16_base, "adapters/sft").merge_and_unload()sft_merged.save_pretrained("sft_merged")tokenizer.save_pretrained("sft_merged")del bf16_base, sft_mergedgc.collect()torch.cuda.empty_cache()

In [ ]:
dpo_base = AutoModelForCausalLM.from_pretrained(    "sft_merged", quantization_config=bnb_config, device_map="auto")dpo_base = prepare_model_for_kbit_training(dpo_base)dpo_lora_config = LoraConfig(    r=16,    lora_alpha=32,    lora_dropout=0.05,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],    task_type="CAUSAL_LM",)

In [ ]:
def build_prompt(question):    return tokenizer.apply_chat_template(        [{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True    )dpo_dataset = Dataset.from_list([    {"prompt": build_prompt(d["prompt"]), "chosen": d["kid"], "rejected": d["adult"]}    for d in kid_adult])

In [ ]:
from trl import DPOConfig, DPOTrainerdpo_config = DPOConfig(    output_dir="adapters/dpo_style",    beta=0.1,    per_device_train_batch_size=2,    gradient_accumulation_steps=8,    num_train_epochs=1,    learning_rate=5e-6,    lr_scheduler_type="cosine",    warmup_ratio=0.03,    logging_steps=20,    save_strategy="no",    fp16=True,    seed=SEED,    report_to="none",    max_length=1024,    max_prompt_length=512,)dpo_trainer = DPOTrainer(    model=dpo_base,    args=dpo_config,    train_dataset=dpo_dataset,    processing_class=tokenizer,    peft_config=dpo_lora_config,)dpo_trainer.train()dpo_trainer.save_model("adapters/dpo_style")

In [ ]:
dpo_model = dpo_trainer.modeldpo_model.eval()generations = generate(dpo_model, prompts)scores = p_simple(generations)print("P_simple (DPO style):", scores.mean())

## Задача 3 — Reward Model

In [ ]:
del dpo_trainer, dpo_base, dpo_modelgc.collect()torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModelForSequenceClassificationif tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenrm_base = AutoModelForSequenceClassification.from_pretrained(    MODEL_ID, num_labels=1, quantization_config=bnb_config, device_map="auto")rm_base.config.pad_token_id = tokenizer.pad_token_idrm_base = prepare_model_for_kbit_training(rm_base)

In [ ]:
def rm_text(instruction, response):    messages = [        {"role": "user", "content": instruction},        {"role": "assistant", "content": response},    ]    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)rm_dataset = Dataset.from_list([    {"chosen": rm_text(d["instruction"], d["chosen"]), "rejected": rm_text(d["instruction"], d["rejected"])}    for d in good_bad])

In [ ]:
from trl import RewardConfig, RewardTrainerrm_lora_config = LoraConfig(    r=16,    lora_alpha=32,    lora_dropout=0.05,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],    modules_to_save=["score"],    task_type="SEQ_CLS",)reward_config = RewardConfig(    output_dir="adapters/reward_model",    per_device_train_batch_size=4,    gradient_accumulation_steps=4,    num_train_epochs=2,    learning_rate=1e-4,    lr_scheduler_type="cosine",    warmup_ratio=0.03,    logging_steps=20,    save_strategy="no",    fp16=True,    seed=SEED,    report_to="none",    max_length=1024,)reward_trainer = RewardTrainer(    model=rm_base,    args=reward_config,    train_dataset=rm_dataset,    processing_class=tokenizer,    peft_config=rm_lora_config,)reward_trainer.train()reward_trainer.save_model("adapters/reward_model")

In [ ]:
rm_model = reward_trainer.modelrm_model.eval()@torch.no_grad()def rm_score(texts):    scores = []    for t in texts:        inputs = tokenizer(t, return_tensors="pt", truncation=True, max_length=1024).to(rm_model.device)        scores.append(rm_model(**inputs).logits.item())    return scorescorrect = 0for d in public_test_quality:    s_chosen, s_rejected = rm_score([        rm_text(d["prompt"], d["chosen"]),        rm_text(d["prompt"], d["rejected"]),    ])    correct += s_chosen > s_rejectedprint("Reward model pairwise accuracy:", correct / len(public_test_quality))